# Open-weights targets on Colab (decisions.md #32)

Runs `collect.py` for the config's vLLM-served models. Use an A100 runtime: the weights are about 16 GB per model, too large for a T4. The answers, explanations and
NLI grouping are produced here; `evaluate.py` and the paper scripts run as usual on the Mac afterwards.

Keys: `HF_TOKEN` (an account granted access to meta-llama/Llama-3.1-8B-Instruct) and `OPENAI_API_KEY` (only used for paraphrases and distractors not yet cached), read from the project's `.env` on Drive, or from Colab secrets (key icon, left sidebar) if not there.
Collections and cache go back to Drive every 10 minutes, so a disconnect loses at most that; re-run from the top to resume.


In [6]:
CONFIG = "configs/smoke.yaml"
MODELS = None        # None = every vLLM model in the config, or a list such as ["qwen3-8b"]
DRIVE_REPO = "/content/drive/MyDrive/Projs/confledger"
REPO = "/content/confledger"
GPU_SHARE = 0.6      # vLLM's share of the GPU; the rest is for the NLI judge
PORT = 8000

In [7]:
# The working copy is local: SQLite is unreliable on the Drive mount. .env stays behind (the next cells read only the keys they need).
import shutil, subprocess, sys, os
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

skip = shutil.ignore_patterns(".git", ".env", ".venv", "docs", "results", "__pycache__", "*.egg-info")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(DRIVE_REPO, REPO, ignore=skip)
cache = Path(REPO, "experiments/results/cache")
cache.mkdir(parents=True)
for db in Path(DRIVE_REPO, "experiments/results/cache").glob("*.sqlite"):
    shutil.copy(db, cache)
os.chdir(REPO)
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-4cb9fe4f-f9e3-1f5a-bb02-039c1efd1e59)



In [8]:
%%bash
# vLLM gets its own venv: its pins conflict with the project's transformers<5. Python 3.12: Colab's
# Python has no ensurepip for `python -m venv`. set -e: a failed install must stop here, not at vLLM start.
set -e
uv venv --allow-existing --python 3.12 --seed /content/vllm_env
/content/vllm_env/bin/pip -q install vllm
python -m pip -q install -e ".[openai,data,nli]"


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment with seed packages at: /content/vllm_env
Activate with: source /content/vllm_env/bin/activate


In [9]:
# Only the two keys this notebook needs, from Drive's .env, else from Colab secrets. Never print them.
from google.colab import userdata

env_file = {}
for line in Path(DRIVE_REPO, ".env").read_text().splitlines() if Path(DRIVE_REPO, ".env").exists() else []:
    key, sep, value = line.strip().removeprefix("export ").partition("=")
    if sep and not key.startswith("#"):
        env_file[key.strip()] = value.strip().strip("'\"")


def key_from(name):
    if env_file.get(name):
        return env_file[name]
    try:
        return userdata.get(name)
    except Exception:
        return None


os.environ["OPENAI_API_KEY"] = key_from("OPENAI_API_KEY") or "EMPTY"   # EMPTY: cached paraphrases only; a cache miss then fails loudly
if key_from("HF_TOKEN"):
    os.environ["HF_TOKEN"] = key_from("HF_TOKEN")
else:
    print("No HF_TOKEN: gated models will fail to download")
print({k: bool(os.environ.get(k)) and os.environ[k] != "EMPTY" for k in ("OPENAI_API_KEY", "HF_TOKEN")})
os.environ["VLLM_API_KEY"] = "EMPTY"


{'OPENAI_API_KEY': True, 'HF_TOKEN': True}


In [10]:
import sqlite3, time, urllib.request, yaml

cfg = yaml.safe_load(open(CONFIG))
served = [m for m in cfg["models"] if "localhost" in m.get("base_url", "") and (MODELS is None or m["model"] in MODELS)]
results = Path(DRIVE_REPO, "experiments/results")
env_dir = results / "all" / Path(CONFIG).stem
env_dir.mkdir(parents=True, exist_ok=True)


def save(mk):
    """Merge the local cache into Drive's (it may have grown on the Mac) and copy this model's collections."""
    for db in Path("experiments/results/cache").glob("*.sqlite"):
        target, tmp = results / "cache" / db.name, Path("/tmp") / db.name
        shutil.copy(target, tmp) if target.exists() else shutil.copy(db, tmp)
        con = sqlite3.connect(tmp, timeout=60)
        con.execute("ATTACH ? AS new", (str(db),))
        con.execute("INSERT OR IGNORE INTO responses SELECT * FROM new.responses")
        con.commit()
        con.close()
        shutil.copy(tmp, target)
    for f in Path("experiments/results").glob(f"*/*/collections/*__{mk}*.jsonl"):
        out = results / f.relative_to("experiments/results")
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, out)


def start_server(m):
    log = open(f"/content/vllm_{m['model']}.log", "w")
    proc = subprocess.Popen(["/content/vllm_env/bin/vllm", "serve", m["repo"], "--revision", m["revision"],
                             "--served-model-name", m["model"], "--port", str(PORT), "--seed", str(cfg["seed"]),
                             "--max-model-len", "4096", "--gpu-memory-utilization", str(GPU_SHARE)],
                            stdout=log, stderr=subprocess.STDOUT,
                            env={**os.environ, "PATH": f"/content/vllm_env/bin:{os.environ['PATH']}"})
    for _ in range(180):
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2)
            return proc
        except OSError:
            time.sleep(10)
    proc.terminate()
    print(open(log.name).read()[-3000:])
    raise RuntimeError(f"vLLM did not start for {m['model']}")


for m in served:
    mk = m["model"]
    server = start_server(m)
    try:
        env = subprocess.run("/content/vllm_env/bin/pip freeze | grep -iE '^(vllm|torch|transformers)=='; nvidia-smi -L",
                             shell=True, capture_output=True, text=True).stdout
        (env_dir / f"colab_env_{mk}.txt").write_text(f"{m['repo']}@{m['revision']}\n{env}")
        run = subprocess.Popen([sys.executable, "experiments/scripts/collect.py", "-c", CONFIG, "-m", mk, "-v"],
                               stdout=open(f"/content/collect_{mk}.log", "w"), stderr=subprocess.STDOUT)
        saved = time.time()
        while run.poll() is None:
            time.sleep(15)                  # poll often: an idle A100 still spends compute units
            if time.time() - saved > 600:
                save(mk)
                saved = time.time()
        save(mk)
        print(mk, "collect exit", run.returncode)
        print("".join(l for l in open(f"/content/collect_{mk}.log") if "confledger.collection" in l)[-1500:])
        if run.returncode:
            raise RuntimeError(f"collect failed for {mk}: see /content/collect_{mk}.log")
    finally:
        server.terminate()
        server.wait()

qwen3-8b collect exit 0
INFO confledger.collection: collection triviaqa__qwen3-8b.jsonl: 20 items, failures={'answer_parse': 0, 'paraphrase': 0, 'judge': 0}
INFO confledger.collection: collection triviaqa__qwen3-8b__dev.jsonl: 5 items, failures={'answer_parse': 0, 'paraphrase': 0, 'judge': 0}
INFO confledger.collection: triviaqa/qwen3-8b: 20 records clustered by local_nli:microsoft/deberta-large-mnli
INFO confledger.collection: triviaqa/qwen3-8b__dev: 5 records clustered by local_nli:microsoft/deberta-large-mnli
INFO confledger.collection: triviaqa/qwen3-8b: target cache hits=0 misses=950

llama-3.1-8b collect exit 0
INFO confledger.collection: collection triviaqa__llama-3.1-8b.jsonl: 20 items, failures={'answer_parse': 2, 'paraphrase': 0, 'judge': 0}
INFO confledger.collection: collection triviaqa__llama-3.1-8b__dev.jsonl: 5 items, failures={'answer_parse': 2, 'paraphrase': 0, 'judge': 0}
INFO confledger.collection: triviaqa/llama-3.1-8b: 20 records clustered by local_nli:microsoft/de

Back on the Mac, once Drive has synced:

    python experiments/scripts/evaluate.py -c configs/smoke.yaml